# Desarrollo Práctica G9

## Inteligencia de Negocios - Componente Práctico Semana 1

### Dominio de negocio: Ciberseguridad global e indicadores digitales por país

En este notebook se desarrolla la práctica correspondiente a la configuración de un entorno ETL utilizando Python, PostgreSQL y Docker.
El dominio seleccionado es ciberseguridad, utilizando fuentes de datos relacionadas con amenazas globales, indicadores de seguridad digital y usuarios de internet por país.

# Instalación de dependencias

In [1]:
!pip install pandas sqlalchemy psycopg2-binary python-dotenv

  Using cached pandas-3.0.3-cp314-cp314-win_amd64.whl.metadata (19 kB)
  Using cached sqlalchemy-2.0.50-cp314-cp314-win_amd64.whl.metadata (9.8 kB)
  Using cached psycopg2_binary-2.9.12-cp314-cp314-win_amd64.whl.metadata (5.1 kB)
  Using cached python_dotenv-1.2.2-py3-none-any.whl.metadata (27 kB)
  Using cached numpy-2.4.6-cp314-cp314-win_amd64.whl.metadata (6.6 kB)
  Using cached tzdata-2026.2-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached greenlet-3.5.1-cp314-cp314-win_amd64.whl.metadata (3.9 kB)
Using cached pandas-3.0.3-cp314-cp314-win_amd64.whl (9.9 MB)
Using cached sqlalchemy-2.0.50-cp314-cp314-win_amd64.whl (2.1 MB)
Using cached psycopg2_binary-2.9.12-cp314-cp314-win_amd64.whl (2.8 MB)
Using cached python_dotenv-1.2.2-py3-none-any.whl (22 kB)
Using cached greenlet-3.5.1-cp314-cp314-win_amd64.whl (239 kB)
Using cached numpy-2.4.6-cp314-cp314-win_amd64.whl (12.5 MB)
Using cached tzdata-2026.2-py2.py3-none-any.whl (349 kB)

   ---------------------------------------- 0/7 [t


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## Importación de dependencias

En esta sección se importan las librerías necesarias para el desarrollo del proceso ETL.
Se utiliza pandas para la lectura y análisis de archivos CSV, python-dotenv para cargar variables de entorno, os para acceder a dichas variables, y SQLAlchemy para establecer la conexión con PostgreSQL.

In [6]:
import pandas as pd
from dotenv import load_dotenv
import os
from sqlalchemy import create_engine

## Manejo de variables de entorno

En esta sección se cargan las credenciales de conexión a PostgreSQL desde el archivo `.env`.
Esto permite evitar escribir credenciales directamente en el código y mantener una configuración más ordenada y segura.

In [7]:
load_dotenv()

DB_USER = os.getenv("POSTGRES_USER")
DB_PASSWORD = os.getenv("POSTGRES_PASSWORD")
DB_NAME = os.getenv("POSTGRES_DB")
DB_HOST = os.getenv("POSTGRES_HOST")
DB_PORT = os.getenv("POSTGRES_PORT")

print("Base de datos:", DB_NAME)
print("Host:", DB_HOST)
print("Puerto:", DB_PORT)
print("Usuario:", DB_USER)

Base de datos: db_ciberseguridad_etl
Host: localhost
Puerto: 5432
Usuario: admin


## Conexión a PostgreSQL

En esta sección se crea la conexión entre Python y la base de datos PostgreSQL alojada en el contenedor Docker.
Para esto se utiliza SQLAlchemy, construyendo una cadena de conexión con las variables cargadas desde el archivo `.env`.

In [8]:
connection_string = f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

engine = create_engine(connection_string)

try:
    with engine.connect() as connection:
        print("Conexión exitosa a PostgreSQL")
except Exception as e:
    print("Error al conectar con PostgreSQL:")
    print(e)~~~~

Conexión exitosa a PostgreSQL


## Fase de extracción de datos

En esta sección se realiza la carga de las fuentes de datos seleccionadas para el proceso ETL.
Se utilizan tres archivos CSV relacionados con el dominio de ciberseguridad: amenazas globales, indicadores de ciberseguridad por país y usuarios de internet por país.

Cada archivo se carga en un DataFrame independiente utilizando la librería pandas.

In [9]:
# Rutas de los archivos CSV
ruta_threats = "data/Global_Cybersecurity_Threats_2015-2024.csv"
ruta_cyber_security = "data/Cyber_security.csv"
ruta_internet_users = "data/internet_users_by_country_cleaned.csv"

# Carga de los archivos CSV en DataFrames
df_threats = pd.read_csv(ruta_threats)
df_cyber_security = pd.read_csv(ruta_cyber_security)
df_internet_users = pd.read_csv(ruta_internet_users)

print("Archivos CSV cargados correctamente.")

Archivos CSV cargados correctamente.


## Validación inicial de los DataFrames

Luego de cargar los archivos CSV, se valida la cantidad de filas y columnas de cada DataFrame.
Esta revisión inicial permite confirmar que las fuentes fueron leídas correctamente antes de continuar con el análisis y la carga hacia PostgreSQL.

In [10]:
print("Dimensiones de df_threats:", df_threats.shape)
print("Dimensiones de df_cyber_security:", df_cyber_security.shape)
print("Dimensiones de df_internet_users:", df_internet_users.shape)

Dimensiones de df_threats: (3000, 10)
Dimensiones de df_cyber_security: (192, 6)
Dimensiones de df_internet_users: (215, 7)


In [11]:
print("Columnas de df_threats:")
print(df_threats.columns)

print("\nColumnas de df_cyber_security:")
print(df_cyber_security.columns)

print("\nColumnas de df_internet_users:")
print(df_internet_users.columns)

Columnas de df_threats:
Index(['Country', 'Year', 'Attack Type', 'Target Industry',
       'Financial Loss (in Million $)', 'Number of Affected Users',
       'Attack Source', 'Security Vulnerability Type',
       'Defense Mechanism Used', 'Incident Resolution Time (in Hours)'],
      dtype='str')

Columnas de df_cyber_security:
Index(['Country', 'Region', 'CEI', 'GCI', 'NCSI', 'DDL'], dtype='str')

Columnas de df_internet_users:
Index(['Country or area', 'Subregion', 'Region', 'Internet users',
       'Population_2021', 'Year', 'Percentage_Internet_Users'],
      dtype='str')


## Vista preliminar de los datos

En esta sección se muestran las primeras cinco filas de cada DataFrame.
Esto permite revisar visualmente la estructura de los datos, los nombres de las columnas y algunos valores de ejemplo.

In [12]:
df_threats.head()

,Country,Year,Attack Type,Target Industry,Financial Loss (in Million $),Number of Affected Users,Attack Source,Security Vulnerability Type,Defense Mechanism Used,Incident Resolution Time (in Hours)
0,China,2019,Phishing,Education,80.53,773169,Hacker Group,Unpatched Software,VPN,63
1,China,2019,Ransomware,Retail,62.19,295961,Hacker Group,Unpatched Software,Firewall,71
2,India,2017,Man-in-the-Middle,IT,38.65,605895,Hacker Group,Weak Passwords,VPN,20
3,UK,2024,Ransomware,Telecommunications,41.44,659320,Nation-state,Social Engineering,AI-based Detection,7
4,Germany,2018,Man-in-the-Middle,IT,74.41,810682,Insider,Social Engineering,VPN,68


In [13]:
df_cyber_security.head()

,Country,Region,CEI,GCI,NCSI,DDL
0,Afghanistan,Asia-Pasific,1.000,5.20,11.69,19.50
1,Albania,Europe,0.566,64.32,62.34,48.74
2,Algeria,Africa,0.721,33.95,33.77,42.81
3,Andorra,Europe,NaN,26.38,NaN,NaN
4,Angola,Africa,NaN,12.99,9.09,22.69


In [14]:
df_internet_users.head()

,Country or area,Subregion,Region,Internet users,Population_2021,Year,Percentage_Internet_Users
0,China,Eastern Asia,Asia,1102140000,1425893465,2022,77.3
1,India,Southern Asia,Asia,881250000,1407563842,2024,62.6
2,United States,Northern America,Americas,311300000,336997624,2023,92.4
3,Indonesia,South-eastern Asia,Asia,215626156,273753191,2023,78.8
4,Pakistan,Southern Asia,Asia,170000000,240000000,2022,70.8


## Carga de datos originales hacia PostgreSQL

En esta sección se cargan los DataFrames originales hacia PostgreSQL.
Estas tablas representan la capa raw del proceso ETL, es decir, los datos tal como fueron extraídos desde los archivos CSV.

Mantener una capa raw permite conservar trazabilidad sobre las fuentes originales antes de aplicar transformaciones o limpieza de datos.

In [15]:
# Carga de los DataFrames originales hacia PostgreSQL como tablas raw

df_threats.to_sql(
    "raw_global_cybersecurity_threats",
    engine,
    if_exists="replace",
    index=False
)

df_cyber_security.to_sql(
    "raw_cyber_security_index",
    engine,
    if_exists="replace",
    index=False
)

df_internet_users.to_sql(
    "raw_internet_users_by_country",
    engine,
    if_exists="replace",
    index=False
)

print("Tablas raw cargadas correctamente en PostgreSQL.")

Tablas raw cargadas correctamente en PostgreSQL.


## Transformación y limpieza de datos

En esta sección se generan copias transformadas de los DataFrames originales.
El objetivo es estandarizar nombres de columnas, homologar nombres de países y preparar los datos para análisis posteriores.

Se mantiene la capa raw en PostgreSQL y se crea una capa clean con datos más consistentes.

In [16]:
# Crear copias para transformación
df_threats_clean = df_threats.copy()
df_cyber_security_clean = df_cyber_security.copy()
df_internet_users_clean = df_internet_users.copy()

In [18]:
# Renombrar columnas del dataset de amenazas globales
df_threats_clean = df_threats_clean.rename(columns={
    "Country": "country",
    "Year": "year",
    "Attack Type": "attack_type",
    "Target Industry": "target_industry",
    "Financial Loss (in Million $)": "financial_loss_million_usd",
    "Number of Affected Users": "affected_users",
    "Attack Source": "attack_source",
    "Security Vulnerability Type": "security_vulnerability_type",
    "Defense Mechanism Used": "defense_mechanism_used",
    "Incident Resolution Time (in Hours)": "incident_resolution_time_hours"
})

# Renombrar columnas del dataset de indicadores de ciberseguridad
df_cyber_security_clean = df_cyber_security_clean.rename(columns={
    "Country": "country",
    "Region": "region",
    "CEI": "cei",
    "GCI": "gci",
    "NCSI": "ncsi",
    "DDL": "ddl"
})

# Renombrar columnas del dataset de usuarios de internet
df_internet_users_clean = df_internet_users_clean.rename(columns={
    "Country or area": "country",
    "Subregion": "subregion",
    "Region": "region",
    "Internet users": "internet_users",
    "Population_2021": "population_2021",
    "Year": "year",
    "Percentage_Internet_Users": "percentage_internet_users"
})

print("Columnas renombradas correctamente.")

Columnas renombradas correctamente.


In [19]:
# Diccionario para homologar nombres de países
country_mapping = {
    "USA": "United States",
    "UK": "United Kingdom"
}

# Aplicar homologación en el dataset principal de amenazas
df_threats_clean["country"] = df_threats_clean["country"].replace(country_mapping)

print("Países homologados correctamente.")

Países homologados correctamente.


In [20]:
print("Valores nulos en df_threats_clean:")
print(df_threats_clean.isnull().sum())

print("\nValores nulos en df_cyber_security_clean:")
print(df_cyber_security_clean.isnull().sum())

print("\nValores nulos en df_internet_users_clean:")
print(df_internet_users_clean.isnull().sum())

Valores nulos en df_threats_clean:
country                           0
year                              0
attack_type                       0
target_industry                   0
financial_loss_million_usd        0
affected_users                    0
attack_source                     0
security_vulnerability_type       0
defense_mechanism_used            0
incident_resolution_time_hours    0
dtype: int64

Valores nulos en df_cyber_security_clean:
country     0
region      0
cei        84
gci         2
ncsi       25
ddl        40
dtype: int64

Valores nulos en df_internet_users_clean:
country                      0
subregion                    0
region                       0
internet_users               0
population_2021              0
year                         0
percentage_internet_users    0
dtype: int64


In [21]:
# Crear columna de control para identificar países con indicadores incompletos
df_cyber_security_clean["has_missing_security_index_data"] = df_cyber_security_clean[
    ["cei", "gci", "ncsi", "ddl"]
].isnull().any(axis=1)

df_cyber_security_clean.head()

,country,region,cei,gci,ncsi,ddl,has_missing_security_index_data
0,Afghanistan,Asia-Pasific,1.000,5.20,11.69,19.50,False
1,Albania,Europe,0.566,64.32,62.34,48.74,False
2,Algeria,Africa,0.721,33.95,33.77,42.81,False
3,Andorra,Europe,NaN,26.38,NaN,NaN,True
4,Angola,Africa,NaN,12.99,9.09,22.69,True


## Carga de datos transformados hacia PostgreSQL

Luego de aplicar las transformaciones necesarias, se cargan los DataFrames limpios hacia PostgreSQL.
Estas tablas representan la capa clean del proceso ETL y contienen datos preparados para el análisis.

In [22]:
# Carga de los DataFrames transformados hacia PostgreSQL como tablas clean

df_threats_clean.to_sql(
    "clean_global_cybersecurity_threats",
    engine,
    if_exists="replace",
    index=False
)

df_cyber_security_clean.to_sql(
    "clean_cyber_security_index",
    engine,
    if_exists="replace",
    index=False
)

df_internet_users_clean.to_sql(
    "clean_internet_users_by_country",
    engine,
    if_exists="replace",
    index=False
)

print("Tablas clean cargadas correctamente en PostgreSQL.")

Tablas clean cargadas correctamente en PostgreSQL.


In [23]:
query = """
SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'public'
ORDER BY table_name;
"""

tables = pd.read_sql(query, engine)
tables

,table_name
0,clean_cyber_security_index
1,clean_global_cybersecurity_threats
2,clean_internet_users_by_country
3,raw_cyber_security_index
4,raw_global_cybersecurity_threats
5,raw_internet_users_by_country


In [24]:
tables_to_check = [
    "raw_global_cybersecurity_threats",
    "raw_cyber_security_index",
    "raw_internet_users_by_country",
    "clean_global_cybersecurity_threats",
    "clean_cyber_security_index",
    "clean_internet_users_by_country"
]

for table in tables_to_check:
    query = f"SELECT COUNT(*) AS total_registros FROM {table};"
    result = pd.read_sql(query, engine)
    print(f"{table}: {result['total_registros'][0]} registros")

raw_global_cybersecurity_threats: 3000 registros
raw_cyber_security_index: 192 registros
raw_internet_users_by_country: 215 registros
clean_global_cybersecurity_threats: 3000 registros
clean_cyber_security_index: 192 registros
clean_internet_users_by_country: 215 registros


## Análisis exploratorio de los DataFrames

En esta sección se realiza el análisis exploratorio de los tres DataFrames principales utilizados en la práctica.
El objetivo es comprender la estructura de cada fuente de datos, validar la existencia de valores nulos y obtener estadísticas básicas de variables numéricas relevantes.

Para cada DataFrame se realizan las siguientes operaciones:
- Mostrar las primeras cinco filas.
- Identificar columnas con valores nulos.
- Contar valores faltantes por columna.
- Calcular promedio, máximo y mínimo de una variable numérica.
- Obtener valores máximos y mínimos de dos variables agrupadas.

## Análisis del DataFrame: Global Cybersecurity Threats

Este DataFrame contiene información sobre incidentes globales de ciberseguridad ocurridos entre 2015 y 2024.
Incluye variables como país, año, tipo de ataque, industria afectada, pérdida financiera, usuarios afectados, vulnerabilidad explotada, mecanismo de defensa y tiempo de resolución del incidente.

In [25]:
# Mostrar las primeras 5 filas del DataFrame de amenazas globales
df_threats_clean.head()

,country,year,attack_type,target_industry,financial_loss_million_usd,affected_users,attack_source,security_vulnerability_type,defense_mechanism_used,incident_resolution_time_hours
0,China,2019,Phishing,Education,80.53,773169,Hacker Group,Unpatched Software,VPN,63
1,China,2019,Ransomware,Retail,62.19,295961,Hacker Group,Unpatched Software,Firewall,71
2,India,2017,Man-in-the-Middle,IT,38.65,605895,Hacker Group,Weak Passwords,VPN,20
3,United Kingdom,2024,Ransomware,Telecommunications,41.44,659320,Nation-state,Social Engineering,AI-based Detection,7
4,Germany,2018,Man-in-the-Middle,IT,74.41,810682,Insider,Social Engineering,VPN,68


In [26]:
# Identificar qué columnas contienen valores nulos
df_threats_clean.isnull().any()

country                           False
year                              False
attack_type                       False
target_industry                   False
financial_loss_million_usd        False
affected_users                    False
attack_source                     False
security_vulnerability_type       False
defense_mechanism_used            False
incident_resolution_time_hours    False
dtype: bool

In [27]:
# Contar cuántos valores faltantes existen en cada columna
df_threats_clean.isnull().sum()

country                           0
year                              0
attack_type                       0
target_industry                   0
financial_loss_million_usd        0
affected_users                    0
attack_source                     0
security_vulnerability_type       0
defense_mechanism_used            0
incident_resolution_time_hours    0
dtype: int64

In [28]:
# Calcular promedio, máximo y mínimo de la pérdida financiera
df_threats_clean["financial_loss_million_usd"].agg(["mean", "max", "min"])

mean    50.49297
max     99.99000
min      0.50000
Name: financial_loss_million_usd, dtype: float64

In [29]:
# Calcular pérdida financiera máxima y mínima agrupada por país
df_threats_clean.groupby("country")["financial_loss_million_usd"].agg(["max", "min"])

,max,min
country,,
Australia,99.99,0.75
Brazil,99.90,1.01
China,99.99,1.05
France,99.78,0.95
Germany,99.98,0.54
India,99.72,0.60
Japan,99.83,1.72
Russia,99.53,0.54
United Kingdom,99.45,0.50


In [30]:
# Calcular usuarios afectados máximos y mínimos agrupados por tipo de ataque
df_threats_clean.groupby("attack_type")["affected_users"].agg(["max", "min"])

,max,min
attack_type,,
DDoS,998937,735
Malware,999542,424
Man-in-the-Middle,998547,984
Phishing,998507,3395
Ransomware,999635,3432
SQL Injection,999545,1579


## Análisis del DataFrame: Cyber Security Index

Este DataFrame contiene indicadores de ciberseguridad por país.
Incluye datos como región, Cybersecurity Exposure Index, Global Cybersecurity Index, National Cyber Security Index y Digital Development Level.

Este conjunto de datos permite complementar el análisis de incidentes con indicadores relacionados con exposición, madurez y desarrollo digital.

In [31]:
# Mostrar las primeras 5 filas del DataFrame de indicadores de ciberseguridad
df_cyber_security_clean.head()

,country,region,cei,gci,ncsi,ddl,has_missing_security_index_data
0,Afghanistan,Asia-Pasific,1.000,5.20,11.69,19.50,False
1,Albania,Europe,0.566,64.32,62.34,48.74,False
2,Algeria,Africa,0.721,33.95,33.77,42.81,False
3,Andorra,Europe,NaN,26.38,NaN,NaN,True
4,Angola,Africa,NaN,12.99,9.09,22.69,True


In [32]:
# Identificar qué columnas contienen valores nulos
df_cyber_security_clean.isnull().any()

country                            False
region                             False
cei                                 True
gci                                 True
ncsi                                True
ddl                                 True
has_missing_security_index_data    False
dtype: bool

In [33]:
# Contar cuántos valores faltantes existen en cada columna
df_cyber_security_clean.isnull().sum()

country                             0
region                              0
cei                                84
gci                                 2
ncsi                               25
ddl                                40
has_missing_security_index_data     0
dtype: int64

In [34]:
# Calcular promedio, máximo y mínimo del Global Cybersecurity Index
df_cyber_security_clean["gci"].agg(["mean", "max", "min"])

mean     52.769526
max     100.000000
min       1.350000
Name: gci, dtype: float64

In [35]:
# Calcular GCI máximo y mínimo agrupado por región
df_cyber_security_clean.groupby("region")["gci"].agg(["max", "min"])

,max,min
region,,
Africa,96.89,1.46
Asia-Pasific,99.54,1.35
Europe,99.54,13.83
North America,100.00,2.20
South America,96.60,16.14


In [36]:
# Calcular NCSI máximo y mínimo agrupado por región
df_cyber_security_clean.groupby("region")["ncsi"].agg(["max", "min"])

,max,min
region,,
Africa,70.13,1.30
Asia-Pasific,84.42,2.60
Europe,96.10,2.60
North America,71.43,3.90
South America,63.64,10.39


## Análisis del DataFrame: Internet Users by Country

Este DataFrame contiene información sobre usuarios de internet, población y porcentaje de penetración de internet por país.
Esta fuente permite analizar el contexto digital de cada país y relacionarlo posteriormente con los incidentes de ciberseguridad.

In [37]:
# Mostrar las primeras 5 filas del DataFrame de usuarios de internet por país
df_internet_users_clean.head()

,country,subregion,region,internet_users,population_2021,year,percentage_internet_users
0,China,Eastern Asia,Asia,1102140000,1425893465,2022,77.3
1,India,Southern Asia,Asia,881250000,1407563842,2024,62.6
2,United States,Northern America,Americas,311300000,336997624,2023,92.4
3,Indonesia,South-eastern Asia,Asia,215626156,273753191,2023,78.8
4,Pakistan,Southern Asia,Asia,170000000,240000000,2022,70.8


In [38]:
# Identificar qué columnas contienen valores nulos
df_internet_users_clean.isnull().any()

country                      False
subregion                    False
region                       False
internet_users               False
population_2021              False
year                         False
percentage_internet_users    False
dtype: bool

In [39]:
# Contar cuántos valores faltantes existen en cada columna
df_internet_users_clean.isnull().sum()

country                      0
subregion                    0
region                       0
internet_users               0
population_2021              0
year                         0
percentage_internet_users    0
dtype: int64

In [40]:
# Calcular promedio, máximo y mínimo del porcentaje de usuarios de internet
df_internet_users_clean["percentage_internet_users"].agg(["mean", "max", "min"])

mean     57.368372
max     100.000000
min       1.700000
Name: percentage_internet_users, dtype: float64

In [41]:
# Calcular usuarios de internet máximos y mínimos agrupados por región
df_internet_users_clean.groupby("region")["internet_users"].agg(["max", "min"])

,max,min
region,,
Africa,136203231,2906
Americas,311300000,2833
Asia,1102140000,275717
Europe,129800000,20100
Oceania,25309515,1034


In [42]:
# Calcular población máxima y mínima agrupada por subregión
df_internet_users_clean.groupby("subregion")["population_2021"].agg(["max", "min"])

,max,min
subregion,,
"Australia, New Zealand",25921089,5129727
Caribbean,11447569,4417
Central America,17608483,400031
Central Asia,34081449,6341855
Eastern Africa,120283026,106471
Eastern Asia,1425893465,686607
Eastern Europe,145102755,3061506
Melanesia,9949437,287800
Micronesia,534606,12511


## Revisión de tipos de datos

En esta sección se revisan los tipos de datos de cada DataFrame.
Esto permite verificar que las variables numéricas, categóricas y temporales hayan sido interpretadas correctamente por pandas.

In [43]:
print("Tipos de datos - Global Cybersecurity Threats")
print(df_threats_clean.dtypes)

print("\nTipos de datos - Cyber Security Index")
print(df_cyber_security_clean.dtypes)

print("\nTipos de datos - Internet Users by Country")
print(df_internet_users_clean.dtypes)

Tipos de datos - Global Cybersecurity Threats
country                               str
year                                int64
attack_type                           str
target_industry                       str
financial_loss_million_usd        float64
affected_users                      int64
attack_source                         str
security_vulnerability_type           str
defense_mechanism_used                str
incident_resolution_time_hours      int64
dtype: object

Tipos de datos - Cyber Security Index
country                                str
region                                 str
cei                                float64
gci                                float64
ncsi                               float64
ddl                                float64
has_missing_security_index_data       bool
dtype: object

Tipos de datos - Internet Users by Country
country                          str
subregion                        str
region                           str
internet_us

## Estadísticas descriptivas generales

A continuación se muestran estadísticas descriptivas de las variables numéricas de cada DataFrame.
Esto permite observar medidas como promedio, desviación estándar, mínimo, máximo y cuartiles.

In [44]:
df_threats_clean.describe()

,year,financial_loss_million_usd,affected_users,incident_resolution_time_hours
count,3000.000000,3000.000000,3000.000000,3000.000000
mean,2019.570333,50.492970,504684.136333,36.476000
std,2.857932,28.791415,289944.084972,20.570768
min,2015.000000,0.500000,424.000000,1.000000
25%,2017.000000,25.757500,255805.250000,19.000000
50%,2020.000000,50.795000,504513.000000,37.000000
75%,2022.000000,75.630000,758088.500000,55.000000
max,2024.000000,99.990000,999635.000000,72.000000


In [45]:
df_cyber_security_clean.describe()

,cei,gci,ncsi,ddl
count,108.000000,190.000000,167.000000,152.000000
mean,0.471861,52.769526,43.306587,51.707829
std,0.217246,34.884013,26.820907,18.283847
min,0.110000,1.350000,1.300000,11.300000
25%,0.279500,18.257500,19.480000,36.532500
50%,0.483000,53.145000,40.260000,51.790000
75%,0.624250,89.187500,64.940000,65.237500
max,1.000000,100.000000,96.100000,82.930000


In [46]:
df_internet_users_clean.describe()

,internet_users,population_2021,year,percentage_internet_users
count,2.150000e+02,2.150000e+02,215.000000,215.000000
mean,2.464109e+07,3.668079e+07,2021.227907,57.368372
std,1.015554e+08,1.418360e+08,0.647903,29.269782
min,1.034000e+03,1.937000e+03,2020.000000,1.700000
25%,3.506630e+05,8.130960e+05,2021.000000,30.250000
50%,2.532059e+06,6.703799e+06,2021.000000,64.100000
75%,1.001494e+07,2.558691e+07,2021.000000,82.000000
max,1.102140e+09,1.425893e+09,2024.000000,100.000000


# Aplicación Profesional de la Práctica

## Paulo Solis

Como desarrollador de software full stack, los conocimientos adquiridos en esta práctica pueden aplicarse directamente en la gestión, integración y análisis de datos provenientes de distintos sistemas internos y externos. En el sector bancario se trabaja con información altamente sensible y estratégica, por lo que es fundamental contar con procesos ordenados, seguros y trazables para el manejo de datos.

En este entorno se generan y gestionan diferentes tipos de datos, como transacciones financieras, información de clientes, registros de autenticación, logs de aplicaciones, eventos de seguridad, alertas de monitoreo, movimientos operativos, reportes regulatorios, indicadores de riesgo y datos provenientes de canales digitales como banca web o aplicaciones móviles. Muchos de estos datos se encuentran distribuidos en distintas bases, archivos, servicios o plataformas, lo que hace necesario integrarlos para obtener una visión más completa del negocio y de la operación tecnológica.

PostgreSQL podría utilizarse como una base de datos relacional para almacenar información estructurada, centralizar registros relevantes y facilitar consultas analíticas. Docker permitiría desplegar entornos controlados y reproducibles para bases de datos, procesos ETL o servicios de análisis, evitando diferencias entre ambientes de desarrollo, pruebas y producción. Python, mediante librerías como pandas y SQLAlchemy, permitiría extraer datos desde archivos CSV, bases de datos o APIs, transformarlos, limpiarlos, validarlos y cargarlos en una base de datos para su posterior análisis.

La implementación de un proceso ETL en una institución bancaria aportaría beneficios importantes, como la automatización de tareas repetitivas, reducción de errores manuales, mejora en la calidad de los datos, integración de múltiples fuentes y disponibilidad de información confiable para la toma de decisiones. Además, permitiría mantener trazabilidad sobre el origen de los datos y aplicar reglas de validación antes de que la información sea utilizada en reportes o sistemas analíticos.

A partir del análisis de la información obtenida, se podrían resolver problemas relacionados con detección de comportamientos anómalos, identificación de fallos recurrentes en sistemas, análisis de tiempos de respuesta, monitoreo de incidentes de seguridad, evaluación de riesgos operativos y generación de reportes para áreas de negocio o cumplimiento. También se podrían tomar decisiones sobre priorización de mejoras técnicas, optimización de procesos internos, fortalecimiento de controles de seguridad y prevención de incidentes que afecten la continuidad del servicio bancario.

En conclusión, esta práctica permite comprender cómo un proceso ETL bien diseñado puede transformar datos dispersos en información útil, confiable y accionable dentro de un entorno profesional exigente como el sector bancario.

# Evidencias de ejecución

En esta sección se presentan las capturas de pantalla que evidencian la correcta configuración del entorno, la ejecución del contenedor Docker, la conexión con PostgreSQL y la carga de datos en la base de datos.

## Evidencia 1: Estructura del proyecto

![Estructura del proyecto](capturas/01_estructura_proyecto.png)

## Evidencia 2: Ejecución de Docker Compose y contenedor activo con docker ps

![Docker Compose Up](capturas/02_docker_compose_up_docker_ps.png)

## Evidencia 3: Contenedor activo en docker desktop

![Docker PS](capturas/03_docker_desktop.png)

## Evidencia 4: Conexión exitosa a PostgreSQL

![Conexión PostgreSQL](capturas/04_conexion_postgresql.png)

## Evidencia 5: Tablas cargadas en PostgreSQL

![Tablas PostgreSQL](capturas/05_tablas_postgresql.png)
![Tablas PostgreSQL](capturas/06_tablas_postgresql.png)
![Tablas PostgreSQL](capturas/07_tablas_postgresql.png)

## Evidencia 6: Análisis exploratorio de DataFrame

![Análisis DataFrame](capturas/08_analisis_dataframe.png)
![Análisis DataFrame](capturas/09_analisis_dataframe.png)
![Análisis DataFrame](capturas/10_analisis_dataframe.png)
![Análisis DataFrame](capturas/11_analisis_dataframe.png)
![Análisis DataFrame](capturas/12_analisis_dataframe.png)
![Análisis DataFrame](capturas/13_analisis_dataframe.png)
![Análisis DataFrame](capturas/14_analisis_dataframe.png)
![Análisis DataFrame](capturas/15_analisis_dataframe.png)
![Análisis DataFrame](capturas/16_analisis_dataframe.png)
![Análisis DataFrame](capturas/17_analisis_dataframe.png)
![Análisis DataFrame](capturas/18_analisis_dataframe.png)
![Análisis DataFrame](capturas/19_analisis_dataframe.png)
![Análisis DataFrame](capturas/20_analisis_dataframe.png)